# 🚀 SVAMITVA Unified DGX Pipeline (V3)
### Professional Performance Optimization for DGX systems (8x A100/H100)

This notebook provides a production-grade entry point for training the full SVAMITVA feature extraction suite on high-performance compute.

### 🎯 Performance Highlights:
- **Dynamic GPU Selection**: Auto-detects and uses the best available GPU(s).
- **AMP Enabled**: Mixed-precision training for 2-3x speedup.
- **High-Throughput IO**: Optimized workers (32) and batch sizes (128).



In [1]:
import os
import sys
import torch
import logging
from pathlib import Path
import importlib

# ── Robust Environment Setup (Server Optimized) ──
def setup_environment():
    cwd = Path(os.getcwd()).resolve()
    
    # 1. Check if we are ALREADY in the repo root
    if (cwd / "train_engine").exists():
        return cwd
        
    # 2. Check if the repo is in a subdirectory named 'MM'
    if (cwd / "MM" / "train_engine").exists():
        repo_root = cwd / "MM"
        os.chdir(str(repo_root))
        return repo_root
        
    # 3. Search upwards (in case we are deep in a subdir)
    curr = cwd
    for _ in range(5):
        if (curr / "train_engine").exists():
            os.chdir(str(curr))
            return curr
        curr = curr.parent
        
    return cwd

REPO_ROOT = setup_environment()

# Ensure REPO_ROOT is at the absolute front of sys.path
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
elif sys.path[0] != str(REPO_ROOT):
    sys.path.remove(str(REPO_ROOT))
    sys.path.insert(0, str(REPO_ROOT))

importlib.invalidate_caches()

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
logging.getLogger("train_engine").setLevel(logging.INFO)

print(f"CWD (Final): {os.getcwd()}")
print(f"REPO_ROOT:   {REPO_ROOT}")

try:
    import train_engine
    print("✅ Success: 'train_engine' is findable.")
    
    # Check if DATA folder exists relative to where we are now
    data_path = Path("DATA") if Path("DATA").exists() else Path("../DATA")
    print(f"📂 Detected Data Path: {data_path.resolve()}")
except ImportError:
    print("❌ Error: 'train_engine' still not found.")


CWD (Final): /jupyter/sods.user04/MM
REPO_ROOT:   /jupyter/sods.user04/MM
✅ Success: 'train_engine' is findable.
📂 Detected Data Path: /jupyter/sods.user04/DATA


In [ ]:
print("Downloading Segformer weights natively from HuggingFace via transformers.")


In [2]:
# ── DGX Training Configuration ──
from train_engine.config import TrainingConfig

# Optimizing for High-Performance Availability
config = TrainingConfig(
    train_dirs=[
        Path("../DATA/MAP1"),
        Path("../DATA/MAP2"),
        Path("../DATA/MAP3"),
        Path("../DATA/MAP4"),
        Path("../DATA/MAP5"),
        Path("../DATA/MAP6"),
        Path("../DATA/MAP7"),
        Path("../DATA/MAP8"),
        Path("../DATA/MAP9")
    ], 
    batch_size=128,              
    num_epochs=150,
    learning_rate=1e-4,          
    num_workers=32,              
    mixed_precision=True,        
    freeze_backbone_epochs=5,
    checkpoint_dir=Path("check"),  # <-- User requested 'check' folder
    sam2_checkpoint=REPO_ROOT / "check" / ""
)

print(f"Training Dirs: {[str(p) for p in config.train_dirs]}")
print(f"Global Batch Size: {config.batch_size}")


Training Dirs: ['../DATA/MAP1', '../DATA/MAP2', '../DATA/MAP3', '../DATA/MAP4', '../DATA/MAP5', '../DATA/MAP6', '../DATA/MAP7', '../DATA/MAP8', '../DATA/MAP9']
Global Batch Size: 128


In [3]:
# ── Data Loaders & Multi-Task Setup ──
from data.dataset import create_dataloaders
from models.model import EnsembleSvamitvaModel
from models.losses import MultiTaskLoss
from train_engine.trainer import Trainer

train_loader, val_loader = create_dataloaders(
    train_dirs=config.train_dirs,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    val_split=0.15,
    image_size=config.tile_size
)

model = EnsembleSvamitvaModel(
    model_name="nvidia/segformer-b4-finetuned-cityscapes-1024-1024",
    num_roof_classes=config.num_roof_classes,
    dropout=0.1
)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=MultiTaskLoss(),
    config=config
)



2026-03-18 09:34:34.733310: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Total CUDA GPUs detected: 8
INFO: Auto-selected primary GPU:1 (most free memory)
INFO: Using device: cuda:1
INFO: TensorBoard logging enabled.
INFO: 📜 Loaded training history (26 entries)


# 🚀 Start Training
try:
    # Note: Trainer automatically selects the best available GPU(s)
    trainer.fit()
except KeyboardInterrupt:
    print("\n⏸️ Training paused by user.")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    import traceback


---
## 📊 Higher Efficiency Options (Terminal-Only)

To fully leverage NvLink/DDP for maximum speed on the DGX at 8x scale:

```bash
torchrun --nproc_per_node=8 train_engine/train_segmentation.py \
    --train_dirs ./data/MAP1 \
    --batch_size 128 --checkpoint_dir check
